# Laya domain fine-tuning (Kaggle T4): comparator row

Laya 0.4.23 trained with its default recipe (micro-batch 8 x accumulation 8, 4 epochs, RLCD loss)
on the same training rows as Kev (`data/splits/laya_domain/train.v2.jsonl`). Protocol:
`docs/specs/laya_feasibility_plan.md`. The local machine could not run it: `laya-train` scores its
internal calibration slice in fixed batches of 16, which exceeded the local GPU memory.

Kaggle: Settings > Accelerator GPU T4 (one is enough), Internet on, Add Input > the private
dataset `group4-kev-data`. Run all, or "Save Version" with "Save & Run All". The run is saved to
`/kaggle/working/checkpoints/laya_s0` and never overwrites an earlier run.

In [ ]:
# ---- parameters ----
RUN_ID = "laya_s0"
SEED = 0
LAYA_VERSION = "0.4.23"
TRAIN_SHA256 = "5234bcedfb1512243c8ec1c10203ce3064db1ad0458a8dee77765cc45cfe5892"  # data/MANIFEST.md
KAGGLE_DATASET = "group4-kev-data"

In [ ]:
import hashlib, json, pathlib, platform, shutil, subprocess, sys
from datetime import datetime, timezone

def now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

found = sorted(pathlib.Path("/kaggle/input").rglob("kev_plan.json"))
assert found, f"attach the dataset {KAGGLE_DATASET} (Add Input > Your Datasets)"
DATA = found[0].parent
# Found by content, wherever it was uploaded (Kaggle datasets are flat: the Laya rows need a file
# name different from the Kev file of the same name, for example laya_train.v2.jsonl).
TRAIN = next((p for p in sorted(DATA.rglob("*.jsonl"))
              if hashlib.sha256(p.read_bytes()).hexdigest() == TRAIN_SHA256), None)
assert TRAIN is not None, f"no .jsonl file under {DATA} has SHA-256 {TRAIN_SHA256[:12]}..."
digest = TRAIN_SHA256
OUT = pathlib.Path("/kaggle/working/checkpoints")
OUT.mkdir(parents=True, exist_ok=True)
target = OUT / RUN_ID
assert not target.exists(), f"{target} exists: a finished run is never overwritten"
WORK = pathlib.Path(f"/tmp/{RUN_ID}")
if WORK.exists():
    shutil.rmtree(WORK)
print("train file ok:", TRAIN)

In [ ]:
# Laya at the pinned version; PyTorch must see the GPU before any training starts.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"laya=={LAYA_VERSION}"], check=True)
import torch, transformers, laya  # noqa: E402
assert torch.cuda.is_available(), "PyTorch cannot use CUDA"
versions = {"python": sys.version.split()[0], "torch": torch.__version__, "cuda": torch.version.cuda,
            "transformers": transformers.__version__, "laya": LAYA_VERSION,
            "gpu": torch.cuda.get_device_name(0)}
print(versions)

In [ ]:
# Train with the library defaults; the log is kept on persistent storage while the run goes on.
command = ["laya-train", "--data", str(TRAIN), "--out", str(WORK), "--device", "cuda", "--seed", str(SEED)]
log_path = OUT / f"{RUN_ID}.train.log"
record = {"run_id": RUN_ID, "platform": "kaggle", **versions, "os": platform.platform(),
          "data_repo_path": "data/splits/laya_domain/train.v2.jsonl", "data_sha256": digest,
          "command": " ".join(command), "started_utc": now()}
print(record["command"])
with open(log_path, "w") as log:
    proc = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT)
record["ended_utc"] = now()
record["exit_code"] = proc.returncode
print(log_path.read_text()[-3000:])

In [ ]:
# Validate and save: a failed run is kept apart and never stored as laya_s0.
required = ("model.safetensors", "questions.json", "train_report.json")
missing = [f for f in required if not (WORK / f).exists()]
ok = record["exit_code"] == 0 and not missing
if not ok:
    target = OUT / f"{RUN_ID}.failed-{datetime.now(timezone.utc):%Y%m%dT%H%M%S}"
    target.mkdir(parents=True)
else:
    shutil.copytree(WORK, target)
shutil.copy(log_path, target / "train.log")
(target / "environment.json").write_text(json.dumps(record, indent=2) + "\n")
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
(target / "pip_freeze.txt").write_text(freeze)
print("saved" if ok else f"FAILED (exit {record['exit_code']}, missing {missing})", "->", target)
if ok:
    print("model.safetensors", hashlib.sha256((target / "model.safetensors").read_bytes()).hexdigest())
assert ok, "the Laya run failed; see train.log"

## After the session

Download `checkpoints/laya_s0` from the version's Output into `models/checkpoints/laya_s0` on the
machine with the repository. Scoring follows `docs/specs/laya_feasibility_plan.md`, Section 3.